In [ ]:
!pip install -U -q transformers flash-linear-attention
%cd /content
!mkdir model
!curl -L -o model/model.safetensors https://huggingface.co/Qwen/Qwen3.5-0.8B/resolve/main/model.safetensors-00001-of-00001.safetensors
!curl -L -o model/tokenizer.json https://huggingface.co/Qwen/Qwen3.5-0.8B/resolve/main/tokenizer.json
!curl -L -o model/config.json https://huggingface.co/Qwen/Qwen3.5-0.8B/resolve/main/config.json
!curl -L -o model/chat_template.jinja https://huggingface.co/Qwen/Qwen3.5-0.8B/resolve/main/chat_template.jinja

In [ ]:
import os
import glob

import torch
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

def load_model(model_dir, dtype, device):
  model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=dtype, local_files_only=True)

  has_lm_head = any(
      "lm_head.weight" in safe_open(f, framework="pt").keys()
      for f in glob.glob(os.path.join(model_dir, "*.safetensors"))
  )

  if not has_lm_head:
    model.lm_head.weight = model.get_input_embeddings().weight

  return model.to(device).eval()

def next_token(logits, temperature):
  if temperature <= 0:
    return logits.argmax(-1, keepdim=True)
  probs = torch.softmax(logits.float() / temperature, dim=-1)
  return torch.multinomial(probs, 1)

@torch.inference_mode()
def generate(model, tokenizer, text, max_new_tokens, temperature, device):
  input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
  cache = DynamicCache(config=model.config)
  n = input_ids.shape[1]

  out = model(input_ids=input_ids, past_key_values=cache, use_cache=True,
              position_ids=torch.arange(n, device=device)[None])
  
  new_ids = []

  for step in range(max_new_tokens):
    tok = next_token(out.logits[:, -1], temperature)
    if tok.item() == tokenizer.eos_token_id:
      break

    new_ids.append(tok.item())
    print(tokenizer.decode(tok[0]), end="", flush=True)

    out = model(input_ids=tok, past_key_values=cache, use_cache=True,
                position_ids=torch.tensor([[n + step]], device=device))
    
  print()
  return new_ids

In [ ]:
import getpass, subprocess
REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "main"
CLONE = "/content/repo"
tok = getpass.getpass("GitHub token, only for a private fork (leave empty otherwise): ")
url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@", 1) if tok else REPO_URL
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CLONE], check=True, capture_output=True)
subprocess.run(["git", "-C", CLONE, "remote", "set-url", "origin", REPO_URL], check=True)

In [ ]:
device, dtype = "cuda", torch.float16
model_dir = "/content/model"
model = load_model(model_dir, device=device, dtype=dtype)
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = "<|endoftext|>"

In [ ]:
import sys, os, types
EVAL = f"{CLONE}/adapter-training/eval"
sys.path.insert(0, EVAL)

from run import HFBackend, new_state, run_hf
from env import tool_definitions, extraction_schema
from common import read_jsonl
import grade

model.eval()
model.config.use_cache = True
backend = HFBackend(model=model, tokenizer=tokenizer)

gold = read_jsonl(f"{EVAL}/data/eval.jsonl")
by_home = {}
for g in gold:
    by_home.setdefault(g["homeLocale"], []).append(g)
gold = [g for rows in by_home.values() for g in rows[:10]]
ledgers = {p["id"]: p["ledger"] for p in read_jsonl(f"{EVAL}/data/eval_profiles.jsonl")}

states = [new_state(g, ledgers[g["profile"]], with_tools=True, with_ledger=True) for g in gold]
args = types.SimpleNamespace(batch=16, tool_tokens=160, answer_tokens=600, max_rounds=4, temperature=0.0)
run_hf(backend, states, args, tool_definitions(with_ledger=True), extraction_schema())

preds = [{"id": s["sample"]["id"], "output": s.get("output", ""), "trail": s["episode"].trail} for s in states]
summary, per_sample = grade.grade(gold, preds)
grade.print_table(summary)

In [ ]:
!cd {CLONE}/adapter-training/eval && python run.py --backend openai --base-url http://127.0.0.1:8090/v1 --model bonsai --no-think \
    --gold data/eval.jsonl --name bonsai-27b --limit 70 --timeout 900

In [ ]:
!cd {CLONE}/adapter-training/eval && python grade.py --gold data/eval.jsonl --pred runs/bonsai-27b/predictions.jsonl --out runs/bonsai-27b/grade.json

In [ ]:
!cd {CLONE}/adapter-training/eval && python run.py --backend openai --base-url http://127.0.0.1:8090/v1 --model bonsai --no-think \
    --gold data/eval.jsonl --name bonsai-27b --limit 70 --timeout 900